In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import optim
from torch.utils.data import Subset
from model import DayCentModel
from data import DayCentDataset
import os
import wandb
from utils import evaluate

In [2]:
# Reproducibility
RND_SEED = 42
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)

In [3]:
INPUT_NPY = "/users/6/mehta423/daycent/data/experiment9/train_X.npy"
OUTPUT_NPY = "/users/6/mehta423/daycent/data/experiment9/train_Y.npy"
INIT_COND = "/users/6/mehta423/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"
OUTPUT_DIR = "/users/6/mehta423/daycent/output/experiment9"

In [4]:
# ----------------------
# Config
# ----------------------
BATCH_SIZE = 2048
EPOCHS = 100
LR = 1e-2
DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    # Set the wandb project where this run will be logged.
    project="daycent",
    name="experiment9/pid-holdout-3quadrants",
    notes="For this the holdout is quadrant wise. The top right is for testing. Rest of the 3 quadrants are for training. Using 10 scenarios.",
    config={
        "learning_rate": LR,
        "architecture": "LSTM with Attention",
        "dataset": "10 Scenarios, 56 Points",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE
    },
)

wandb: Currently logged in as: dwij (dwij-university-of-minnesota) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)

0


In [6]:
len(dataset)

37000

In [7]:
# 1) Define split boundaries
UNIT_SIZE = 3700  # Each point_id has 3700 samples (10 scenarios * 56 points * 25 years)
train_size = UNIT_SIZE * 7
val_size = UNIT_SIZE * 2    
test_size = UNIT_SIZE * 1       
# Total: 25000

# 2) Create index arrays for each split
train_idx = np.arange(0, train_size)
val_idx = np.arange(train_size, train_size + val_size)
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)

# 3) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 4) Create loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

Dataset sizes — total: 37000, train: 25900, val: 7400, test: 3700


In [8]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")

Input feature dim: 20, init cond dim: 245, year enc dim: 16


In [9]:

model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.to(DEVICE)

DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=20, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [10]:
# ----------------------
# Optimizer & scheduler
# ----------------------
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

In [11]:
import matplotlib.pyplot as plt

# ----------------------
# Training loop with loss tracking
# ----------------------
best_val_loss = float('inf')


# Initialize loss tracking lists
train_losses = []
val_somsc_losses = []
val_yield_losses = []
val_total_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        # move to device
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        optimizer.zero_grad()
        out = model(batch)

        # --- SOMSC loss ---
        somsc_target = batch["somsc"]              # (B,12)
        somsc_mask = batch["somsc_mask"]           # (B,12)

        # compute masked MSE
        somsc_loss = ((out["somsc_pred"].squeeze(-1) - somsc_target)**2 * somsc_mask).sum() / somsc_mask.sum()

        # --- Yield loss ---
        yield_target = batch["yield"]              # (B,)
        yield_mask = batch["yield_mask"]           # (B,)
        yield_loss = ((out["yield_pred"] - yield_target)**2 * yield_mask).sum() / yield_mask.sum()

        # --- total loss ---
        alpha = 1.0  # weight for SOMSC loss
        beta = 1.0   # weight for Yield loss
        loss = alpha*somsc_loss + beta*yield_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["sequence"].size(0)

    total_loss /= len(dataset)
    train_losses.append(total_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")
    somsc_loss_val, yield_loss_val = evaluate(model, val_loader, DEVICE)
    print(f"  Val SOMSC Loss: {somsc_loss_val:.4f}, Yield Loss: {yield_loss_val:.4f}")

    if yield_loss_val < best_val_loss:
        best_val_loss = yield_loss_val
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"Saved best model at epoch {epoch} with val_loss: {yield_loss_val:.4f}")
    
    # Track validation losses
    val_somsc_losses.append(somsc_loss_val)
    val_yield_losses.append(yield_loss_val)
    val_total_losses.append(somsc_loss_val + yield_loss_val)

    run.log({
        "epoch": epoch + 1,
        "train_loss": total_loss,
        "val_somsc_loss": somsc_loss_val,
        "val_yield_loss": yield_loss_val,
        "val_total_loss": somsc_loss_val + yield_loss_val,
        "learning_rate": optimizer.param_groups[0]['lr'],
    })

    scheduler.step(total_loss)


Epoch 1/100 - Loss: 1.7941
  Val SOMSC Loss: 0.1848, Yield Loss: 1.1053
Saved best model at epoch 0 with val_loss: 1.1053
Epoch 2/100 - Loss: 0.7031
  Val SOMSC Loss: 0.0902, Yield Loss: 1.0488
Saved best model at epoch 1 with val_loss: 1.0488
Epoch 3/100 - Loss: 0.6454
  Val SOMSC Loss: 0.0834, Yield Loss: 0.9778
Saved best model at epoch 2 with val_loss: 0.9778
Epoch 4/100 - Loss: 0.6045
  Val SOMSC Loss: 0.0677, Yield Loss: 0.9564
Saved best model at epoch 3 with val_loss: 0.9564
Epoch 5/100 - Loss: 0.5829
  Val SOMSC Loss: 0.0563, Yield Loss: 0.9028
Saved best model at epoch 4 with val_loss: 0.9028
Epoch 6/100 - Loss: 0.5772
  Val SOMSC Loss: 0.0592, Yield Loss: 0.8819
Saved best model at epoch 5 with val_loss: 0.8819
Epoch 7/100 - Loss: 0.5569
  Val SOMSC Loss: 0.0472, Yield Loss: 0.9031
Epoch 8/100 - Loss: 0.5414
  Val SOMSC Loss: 0.0632, Yield Loss: 0.9645
Epoch 9/100 - Loss: 0.5576
  Val SOMSC Loss: 0.0497, Yield Loss: 0.8489
Saved best model at epoch 8 with val_loss: 0.8489
Ep

In [12]:
evaluate(model, test_loader, DEVICE)

(0.1368027026749946, 0.08578192208264325)

In [13]:
run.finish()


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▇▇▇▆▆▆▄▄▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_somsc_loss,█▂▂▃▄▂▁▂▁▁▁▂▁▂▁▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_total_loss,▇▇▇█▇▅▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_yield_loss,█▇▇▇█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
learning_rate,0.01
train_loss,0.05141
val_somsc_loss,0.04192
val_total_loss,0.0987
